# TeleRAG: RAG Generation Evaluation (v4)

**What this does:** Evaluates our fine-tuned model + RAG context on TeleQnA questions.

**Key fixes in v4:**
- Correct Llama-3 chat prompt template
- Robust answer extraction supporting A-E and all model output formats
- Real 3GPP context from 35K chunks (not boilerplate)
- Loads LoRA adapter directly with Unsloth
- Tests on 10 first (sanity), then configurable for 50+

**Prerequisites:**
- Upload `kaggle_eval_input.json` as a Kaggle dataset
- Have your LoRA adapter on HuggingFace

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers datasets trl accelerate bitsandbytes xformers

In [ ]:
# Cell 2: Load Model
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import re
import torch
from unsloth import FastLanguageModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

max_seq_length = 4096
load_in_4bit = True

# ===== TOGGLE: Switch between base model and LoRA adapter =====
USE_LORA = True  # Set to False to test base model only

BASE_MODEL = "AliMaatouk/LLama-3-8B-Tele-it"
ADAPTER_REPO = "Imchaitanya/TeleRAG_LoRA"

# ===== CHANGE THIS to your evaluation dataset path =====
EVAL_DATA_PATH = "/kaggle/input/YOUR_DATASET/kaggle_eval_golden_final.json"

# Login
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in with HF token.")
except:
    hf_token = None
    print("No HF token found - model must be public.")

if USE_LORA:
    print(f"Loading LoRA adapter: {ADAPTER_REPO}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_REPO,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
        token=hf_token,
    )
else:
    print(f"Loading BASE model: {BASE_MODEL}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
        token=hf_token,
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)
print(f"Model loaded. USE_LORA={USE_LORA}")


In [ ]:
# Cell 3: Prompt Template + Answer Extraction
#
# CRITICAL: LLama-3-8B-Tele-it uses 'Prompt: ...\n Model:' format
# NOT the standard Llama-3 chat template with special tokens.
# Also: TeleQnA uses 'option 1', 'option 2' numbering, not A/B/C/D/E.
#
import re

def compute_relevance(question, context):
    """Check if context is actually relevant to the question."""
    if not context or len(context) < 100:
        return False
    if context.count('References') >= 3:
        return False
    if context.count('Definitions, symbols and abbreviations') >= 2:
        return False
    stop_words = {'the','a','an','is','of','in','to','and','for','what','which','how',
                  'are','by','on','that','with','this','from','be','as','it','or','at'}
    q_words = set(question.lower().split()) - stop_words
    ctx_lower = context[:1000].lower()
    overlap = sum(1 for w in q_words if len(w) > 3 and w in ctx_lower)
    return overlap >= 2


def format_prompt(item):
    """
    Build prompt in the EXACT format LLama-3-8B-Tele-it expects.
    Model card format: 'Prompt: [instruction]\n'
    
    Options use 'option 1:', 'option 2:' numbering (matching TeleQnA native format).
    """
    options_list = item['options']
    # Use 'option N:' format (matching TeleQnA and training data)
    options_str = '\n'.join(
        f"option {i+1}: {opt}" for i, opt in enumerate(options_list)
    )
    
    context = item.get('context', '')
    context_relevant = compute_relevance(item['question'], context)
    
    # Build instruction in the format the model expects
    instruction = 'Answer the following telecom question by selecting the correct option.'
    
    if context_relevant:
        instruction += f'\n\nReference material:\n{context[:6000]}'
    
    instruction += f'\n\nQuestion: {item["question"]}\n\n{options_str}'
    
    # Model card format: Prompt: [instruction]\n
    prompt = f"Prompt: {instruction}\nModel: "
    return prompt


def extract_answer_letter(response_text, num_options):
    """
    Extract the answer from model output.
    The model outputs in format: 'option N: text'
    We convert to letter (A/B/C/D/E) for comparison with ground truth.
    """
    text = response_text.strip()
    max_letter = chr(64 + num_options)
    
    # Strategy 1: 'option N' (the primary format this model uses)
    match = re.search(r'[Oo]ption\s*(\d+)', text)
    if match:
        opt_num = int(match.group(1))
        if 1 <= opt_num <= num_options:
            return chr(64 + opt_num)
    
    # Strategy 2: Direct letter 'A', 'B', etc.
    match2 = re.search(rf'\b([A-{max_letter}])\b', text[:50])
    if match2:
        return match2.group(1).upper()
    
    # Strategy 3: 'Answer: X' format
    match3 = re.search(rf'Answer:\s*([A-{max_letter}])', text, re.IGNORECASE)
    if match3:
        return match3.group(1).upper()
    
    return 'A'  # fallback

print('Prompt template and answer extraction ready.')

# Preview a prompt


In [ ]:
# ===== REPLACE with your actual Kaggle dataset path =====
DATASET_PATH = "/kaggle/input/datasets/chaitanyakadupukutla/final-rag/kaggle_eval_input.json"

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data)} test questions.")

# Preview first prompt
sample = eval_data[0]
print(f"\nSample question: {sample['question'][:80]}...")
print(f"Options: {len(sample['options'])} | Answer: {sample['answer']}")
print(f"Context: {len(sample['context'])} chars")

# Check context quality
boilerplate_count = sum(
    1 for d in eval_data 
    if d['context'].count('References') >= 3 or d['context'].count('Definitions, symbols') >= 2
)
print(f"\nQuestions with boilerplate context (will use model knowledge instead): {boilerplate_count}/{len(eval_data)}")

# Show the actual prompt for debugging
print(f"\n--- PROMPT PREVIEW ---")
p = format_prompt(sample)
print(p[:600])
print("...")

In [ ]:
# Cell 5: Run Evaluation
from tqdm import tqdm

correct = 0
results = []

for item in tqdm(eval_data):
    prompt = format_prompt(item)
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=3800
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the NEW tokens
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()
    
    # Extract answer
    num_options = len(item["options"])
    predicted = extract_answer_letter(response, num_options)
    actual = item["answer"].strip().upper()
    
    is_correct = (predicted == actual)
    if is_correct:
        correct += 1
    
    results.append({
        "question": item["question"],
        "predicted": predicted,
        "actual": actual,
        "correct": is_correct,
        "full_response": response
    })
    
    # Print first 5 for debugging
    if len(results) <= 5:
        tag = "CORRECT" if is_correct else "WRONG"
        print(f"\n[{tag}] Q: {item['question'][:60]}...")
        print(f"  Pred: {predicted} | Actual: {actual}")
        print(f"  Raw: {response[:100]}")

accuracy = correct / len(eval_data) * 100
print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Total:    {len(eval_data)}")
print(f"Correct:  {correct}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*50}")

# Save results
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved.")

# Quick analysis
d_count = sum(1 for r in results if r['predicted'] == 'D')
empty = sum(1 for r in results if r['predicted'] == '')
print(f"\nDiagnostics: Predicted D: {d_count} | Empty: {empty}")
